<a href="https://colab.research.google.com/github/ghroyd1110/Git-Commands/blob/master/GeneratorExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 42.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import fitz # !pip install pymupdf
from google.colab import drive

# 1. MOUNT DRIVE
drive.mount('/content/drive')

# 2. CONFIGURATION
ROOT_FOLDER = '/content/drive/MyDrive/Permits'
LOG_FILE = '/content/drive/MyDrive/PermitsOutput/generator_data_log.txt'

def get_text_to_right(page, label, width=400):
    rects = page.search_for(label)
    if not rects: return "Not Found"
    target_rect = rects[0]
    search_zone = fitz.Rect(target_rect.x1, target_rect.y0 - 2, target_rect.x1 + width, target_rect.y1 + 2)
    return page.get_text("text", clip=search_zone).strip() or "Not Found"

def extract_adeq_high_volume(pdf_path):
    records = []
    try:
        doc = fitz.open(pdf_path)
        page1 = doc[0]

        # Header Extraction (Geographic)
        p_no_raw = get_text_to_right(page1, "PERMIT No.")
        if p_no_raw == "Not Found": p_no_raw = get_text_to_right(page1, "PERMIT")
        permit_no = re.search(r"(\d+)", p_no_raw).group(1) if re.search(r"(\d+)", p_no_raw) else p_no_raw

        header_data = {
            "Permit No.": permit_no,
            "PERMITTEE": get_text_to_right(page1, "PERMITTEE:"),
            "FACILITY": get_text_to_right(page1, "FACILITY:"),
            "PLACE ID": get_text_to_right(page1, "PLACE ID:"),
            "DATE ISSUED": get_text_to_right(page1, "DATE ISSUED:"),
            "EXPIRY DATE": get_text_to_right(page1, "EXPIRY DATE:"),
            "Source File": os.path.basename(pdf_path)
        }

        # Scan ALL pages for Equipment Tables
        for page in doc:
            p_text = page.get_text()
            # Look for ANY equipment table markers
            if any(x in p_text.upper() for x in ["EQUIPMENT LIST", "ATTACHMENT C", "ATTACHMENT \"C\""]):
                words = page.get_text("words")
                # Sort: Y coordinate (top-to-bottom) then X coordinate (left-to-right)
                words.sort(key=lambda w: (w[3], w[0]))

                lines = []
                if words:
                    curr_y, curr_line = words[0][3], []
                    for w in words:
                        if abs(w[3] - curr_y) < 5: curr_line.append(w[4])
                        else:
                            lines.append(" ".join(curr_line))
                            curr_line, curr_y = [w[4]], w[3]
                    lines.append(" ".join(curr_line))

                # Capture ALL engines and generators
                for i, line in enumerate(lines):
                    # Expanded keyword check
                    is_gen = any(k in line.lower() for k in ["emergency generator", "emergency engine", "internal combustion engine"])
                    if is_gen:
                        try:
                            entry = header_data.copy()
                            # Standard ADEQ offset: Type is index i, data follows in i+1 to i+7
                            entry.update({
                                "Equip Type": line,
                                "Max. Capacity": lines[i+1] if i+1 < len(lines) else "N/A",
                                "Make": lines[i+2] if i+2 < len(lines) else "N/A",
                                "Model": lines[i+3] if i+3 < len(lines) else "N/A",
                                "Serial Number": lines[i+4] if i+4 < len(lines) else "N/A",
                                "Installation/MFG. Date": lines[i+5] if i+5 < len(lines) else "N/A",
                                "Equipment ID Number": lines[i+6] if i+6 < len(lines) else "N/A",
                                "Regulations": lines[i+7] if i+7 < len(lines) else "N/A"
                            })
                            records.append(entry)
                        except: pass
        doc.close()
    except Exception as e: print(f"Error in {pdf_path}: {e}")
    return records

# 3. RUNTIME
with open(LOG_FILE, 'a') as f:
    for root, dirs, files in os.walk(ROOT_FOLDER):
        for file in files:
            if file.lower().endswith(".pdf") and ("final" in file.lower() or "final" in root.lower()):
                results = extract_adeq_high_volume(os.path.join(root, file))
                for r in results:
                    f.write(json.dumps(r) + "\n")
                if results: print(f"Logged {len(results)} units from {file}")

print(f"Extraction complete. Use the conversion script to create your final CSV.")

Mounted at /content/drive
Logged 3 units from Final Permit.pdf
Logged 4 units from 34918 Bowie Draft 02-17-06 R5.pdf
Logged 1 units from Permit 2869_61522.pdf
Logged 2 units from Permit #73015.pdf
Logged 4 units from rpt_calportland_renewal_final_20201027.pdf
Logged 3 units from 2869_85424 Permit(Final).pdf
Logged 3 units from 2869_61522(85849) Permit(Final).pdf
Logged 3 units from 2869_85424(94735) Permit(Final).pdf
Logged 3 units from 2869_85424(96502) Permit(final).pdf
Logged 1 units from 2869_85424(97981) TSD_final.pdf
Logged 1 units from APCC 38592 Final Permit (12-16-08).pdf
Logged 1 units from 1978_78863(103225) Permit.pdf
Logged 1 units from 2025-0110 Carlota Class II Permit Renewal Application_v4.0 FINAL_upated.pdf
Logged 1 units from 1978_107892 Permit(draft)_ final .pdf
Logged 1 units from 1978_107892 Permit_ final .pdf
Logged 1 units from 1978_78863_Permit(Final).pdf
Logged 1 units from 1978_78863_TSD(Final).pdf
Logged 1 units from 1978_78863_Permit(Final).pdf
Logged 1 unit

In [ ]:
import pandas as pd
import json
import csv
import os
from google.colab import drive

# 1. MOUNT DRIVE (if not already mounted)
drive.mount('/content/drive')

# 2. CONFIGURATION - Update these to match your exact file names
LOG_FILE = '/content/drive/MyDrive/PermitsOutput/generator_data_log.txt'
FINAL_CSV = '/content/drive/MyDrive/PermitsOutput/Emergency_Generator_Final_Master_2_step.csv'

# 3. CONVERSION LOGIC
data = []

print(f"Reading data from {LOG_FILE}...")

if os.path.exists(LOG_FILE):
    with open(LOG_FILE, 'r') as f:
        for line_num, line in enumerate(f, 1):
            try:
                # Load each line as a JSON object
                item = json.loads(line)
                data.append(item)
            except Exception as e:
                print(f"Skipping line {line_num} due to formatting error: {e}")

    if data:
        # Convert the list of dictionaries into a Table (DataFrame)
        df = pd.DataFrame(data)

        # CLEANUP: Remove exact duplicates (if the script was run twice on the same folder)
        # We define a duplicate as having the same Permit No and Equipment ID
        if "Permit No." in df.columns and "Equipment ID Number" in df.columns:
            initial_count = len(df)
            df = df.drop_duplicates(subset=["Permit No.", "Equipment ID Number"])
            print(f"Removed {initial_count - len(df)} duplicate entries.")

        # SAVE TO CSV WITH HEADERS AND DOUBLE QUOTES
        # quoting=csv.QUOTE_ALL ensures every field is wrapped in ""
        df.to_csv(FINAL_CSV, index=False, quoting=csv.QUOTE_ALL)

        print("-" * 30)
        print(f"SUCCESS!")
        print(f"Total Records Processed: {len(df)}")
        print(f"Final CSV Created: {FINAL_CSV}")
        print("-" * 30)
    else:
        print("The log file was empty. No data to convert.")
else:
    print(f"Error: Could not find {LOG_FILE}. Please check the file path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading data from /content/drive/MyDrive/PermitsOutput/generator_data_log.txt...
Removed 447 duplicate entries.
------------------------------
SUCCESS!
Total Records Processed: 318
Final CSV Created: /content/drive/MyDrive/PermitsOutput/Emergency_Generator_Final_Master_2_step.csv
------------------------------
